# 📥 Guia de Importação de Planilhas - Centauro ERP

Este notebook documenta como o sistema importa dados de planilhas externas para:
1. **Combustível de Veículos** - Relatório de Resumo de Consumo (Ticket Log)
2. **Folha de Pagamento** - Controle de Custos de Mão de Obra (Luiza)

---

## 1. 🚗 Importação de Combustível de Veículos

### Origem do Arquivo
O arquivo é obtido na plataforma **Ticket Log** (plataforma.ticketlog.com.br):
- Acessar: **Geração e Download de Relatórios**
- Selecionar: **RESUMO DE CONSUMO**
- O relatório contém litros abastecidos e valores por cartão/veículo

### Nome do Arquivo
O arquivo geralmente é salvo como:
- `RelResumidoConsumo.xls`

### Formato do Arquivo
O arquivo é um **HTML disfarçado de Excel** (.xls que na verdade é HTML).

### Estrutura da Planilha

| Linha | Conteúdo |
|-------|----------|
| 1-4 | Cabeçalhos e informações do relatório (período, etc.) |
| 5 | Cabeçalho das colunas |
| 6+ | Dados dos veículos |

### Colunas Utilizadas

O sistema lê as seguintes colunas (índice baseado em 0):

| Índice | Coluna | Descrição | Exemplo |
|--------|--------|-----------|----------|
| **0** | Placa | Placa do veículo | `LLV7582` |
| **9** | Última Km | Odômetro no momento do abastecimento | `322.998` |
| **10** | Km Rodados | Quilômetros rodados no período | `1.607` |
| **12** | Litros | Total de litros abastecidos | `156,80` |
| **16** | Total (R$) | Valor total gasto | `1.010,95` |

### Cabeçalho Visual da Planilha
```
Placa | Modelo | Família | Ano | Marca | Nr.Frota | Tipo Combustível | Cartão R$ Limite | Saldo | Última Km e/ou H | Km Rodados | Horas Trabalhadas | Litros | Valor Médio (R$) | Litro | Km por Litro | Litros/Hora | Total (R$)
```

In [ ]:
# Exemplo de como o sistema lê o arquivo de combustível
import pandas as pd
import io

# Índices das colunas relevantes
col_placa = 0        # Placa do veículo
col_ultima_km = 9    # Última Km registrada
col_km_rodados = 10  # Km rodados no período
col_litros = 12      # Total de litros
col_total = 16       # Valor total em R$

# Leitura do arquivo (como HTML)
tables = pd.read_html('RelResumidoConsumo.xls')
df = tables[0]

# Pula as 5 primeiras linhas (cabeçalhos)
df = df.iloc[5:]

print("Colunas disponíveis:")
print(df.columns.tolist())

### Tratamento de Números no Formato Brasileiro

O sistema converte automaticamente números do formato brasileiro:

| Entrada | Saída | Explicação |
|---------|-------|------------|
| `1.607` | `1607` | Ponto como separador de milhar |
| `156,80` | `156.80` | Vírgula como separador decimal |
| `1.010,95` | `1010.95` | Combinação de ambos |

A função `safe_int` e `safe_float` detectam automaticamente o formato.

### Agregação de Duplicatas

Quando um veículo aparece **múltiplas vezes** na planilha (ex: abasteceu gasolina e etanol no mesmo mês), o sistema:

1. **Agrupa** por placa
2. **Soma** os valores de:
   - Litros
   - Km Rodados
   - Valor Total
3. **Mantém o maior** valor de Última Km (odômetro)

---

## 2. 👥 Importação de Folha de Pagamento

### Origem do Arquivo
O arquivo é gerado pela **Luiza** no Google Sheets:
- Nome: **[Centauro] Controle de Custos - Mão de obra - Folha de Pgto. Horizontal**

### Formato do Arquivo
Arquivo Excel padrão (`.xlsx`)

### Estrutura da Planilha

A planilha contém os custos mensais de cada colaborador.

### Colunas Utilizadas

O sistema lê as seguintes colunas (índice baseado em 0):

| Índice | Coluna | Descrição | Exemplo |
|--------|--------|-----------|----------|
| **3** | **Custo Total** (D) | Custo total do colaborador no mês | `8.500,00` |
| **6** | **Matrícula** (G) | Número de matrícula do colaborador | `12345` |

### Observações Importantes

1. **Matrícula é a chave** - O sistema usa a matrícula para identificar o colaborador
2. A matrícula deve estar **cadastrada previamente** no sistema
3. Colaboradores sem matrícula correspondente serão **ignorados**

In [ ]:
# Exemplo de como o sistema lê o arquivo de folha de pagamento
import pandas as pd

# Índices das colunas
col_custo_total = 3   # Coluna D - Custo Total
col_matricula = 6     # Coluna G - Matrícula

# Leitura do arquivo Excel
df = pd.read_excel('folha_pagamento.xlsx')

# Processamento de cada linha
for index, row in df.iterrows():
    custo_total = row.iloc[col_custo_total]       # Coluna D
    matricula = row.iloc[col_matricula]          # Coluna G
    
    print(f"Matrícula: {matricula}, Custo: R$ {custo_total}")

### Lógica de Rateio de Custos

Após importar o custo total do colaborador, o sistema:

1. **Busca as alocações** do colaborador no mês correspondente
2. **Calcula a diária** = Custo Total ÷ Total de dias alocados
3. **Distribui o custo** para cada projeto proporcionalmente aos dias trabalhados

#### Exemplo:

| Colaborador | Custo Mensal | Dias Alocados | Diária |
|-------------|--------------|---------------|--------|
| João Silva | R$ 10.000,00 | 20 dias | R$ 500,00 |

| Projeto | Dias | Custo Alocado |
|---------|------|---------------|
| Projeto A | 12 dias | R$ 6.000,00 |
| Projeto B | 8 dias | R$ 4.000,00 |
| **Total** | **20 dias** | **R$ 10.000,00** |

### Custos Não Alocados

Se o colaborador não tiver **nenhuma alocação** no mês:
- O custo total é marcado como **"Não Alocado"**
- Aparece no dashboard financeiro como custo sem projeto associado

---

## 📋 Resumo das Colunas

### Combustível (RelResumidoConsumo.xls)
| Coluna | Índice | Uso |
|--------|--------|-----|
| Placa | 0 | Identificar veículo |
| Última Km | 9 | Atualizar odômetro |
| Km Rodados | 10 | Registro mensal |
| Litros | 12 | Consumo mensal |
| Total (R$) | 16 | Custo mensal |

### Folha de Pagamento
| Coluna | Índice | Uso |
|--------|--------|-----|
| Custo Total | 3 (D) | Custo do colaborador |
| Matrícula | 6 (G) | Identificar colaborador |

## ⚠️ Avisos Importantes

1. **Veículos devem estar cadastrados** antes de importar combustível
2. **Colaboradores devem ter matrícula** cadastrada antes de importar folha
3. Reimportar um mês **sobrescreve** os dados anteriores
4. O sistema detecta automaticamente o **período/mês** do relatório de combustível
5. Para folha de pagamento, o mês/ano deve ser **informado manualmente** no upload